In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/astana_rent_clean.csv")

print(df.shape)

train, test = train_test_split(
    df,
    test_size=0.25,
    random_state=42
)

print(train.shape, test.shape)

(600, 15)
(450, 15) (150, 15)


In [2]:
from sklearn.linear_model import LinearRegression
tr = train.dropna(subset=["area"])
X = np.column_stack([np.ones(len(tr)), tr["area"]])
y = tr["price"].to_numpy()
w = np.linalg.solve(X.T @ X, X.T @ y) # (XᵀX) w = Xᵀy
print(w.round(1))
lr = LinearRegression().fit(tr[["area"]], y)
print(round(lr.intercept_, 1), lr.coef_.round(1))

[9122.1 4265.1]
9122.1 [4265.1]


In [3]:
from sklearn.metrics import mean_squared_error, r2_score

mse = np.square(X @ w - y).mean()
print(round(mse), round(np.sqrt(mse)))

te = test.dropna(subset=["area"])
pred = lr.predict(te[["area"]])
print("test RMSE", round(np.sqrt(mean_squared_error(te["price"], pred))),
      "R2", round(r2_score(te["price"], pred), 3))

2213187001 47045
test RMSE 47198 R2 0.756


In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["rooms", "area", "floor", "floors", "year"]
cat_cols = ["district", "furniture"]
X_train, y_train = train[num_cols + cat_cols], train["price"]
X_test, y_test = test[num_cols + cat_cols], test["price"]

def make_prep(num=num_cols):
    return make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore")), cat_cols))

model = make_pipeline(make_prep(), LinearRegression())
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
print(scores.round(3), scores.mean().round(3))
model.fit(X_train, y_train)
print("test R2", round(model.score(X_test, y_test), 3))

names = [n.split("__")[1] for n in model[:-1].get_feature_names_out()]
coefs = pd.Series(model[-1].coef_, index=names)
print(coefs.round(0).sort_values())

[0.931 0.904 0.89  0.924 0.933] 0.916
test R2 0.914
district_Алматы     -32274.0
district_Байконур   -23170.0
district_Сарыарка   -13846.0
furniture_no         -5736.0
floor                -1128.0
rooms                 1215.0
floors                1287.0
furniture_yes         5736.0
district_Нура        11697.0
year                 14562.0
district_Есиль       57594.0
area                 85605.0
dtype: float64


In [5]:
from sklearn.preprocessing import KBinsDiscretizer

num_wo = ["rooms", "area", "floor", "floors"]

def make_prep_bins(n_bins=8):
    return make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_wo),
        (make_pipeline(SimpleImputer(strategy="median"),
                       KBinsDiscretizer(n_bins=n_bins, encode="onehot", strategy="uniform")),
         ["year"]),
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore")), cat_cols))

model_b = make_pipeline(make_prep_bins(), LinearRegression())
scores = cross_val_score(model_b, X_train, y_train, cv=5, scoring="r2")
print(scores.round(3), scores.mean().round(3))
model_b.fit(X_train, y_train)
print("test", round(model_b.score(X_test, y_test), 3))

[0.928 0.903 0.887 0.923 0.931] 0.914
test 0.911


In [6]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

small = train.dropna(subset=["area", "year"]).sample(30, random_state=1).copy()
small["price_m2"] = small["price"] / small["area"]
big = test.dropna(subset=["area", "year"]).copy()
big["price_m2"] = big["price"] / big["area"]
xs, ys = small[["year"]], small["price_m2"]

def rmse(m, X, y):
    return round(np.sqrt(mean_squared_error(y, m.predict(X))))

def poly(deg, reg):
    return make_pipeline(PolynomialFeatures(deg, include_bias=False),
                         StandardScaler(), reg)

for deg in [1, 4, 15]:
    m = poly(deg, LinearRegression()).fit(xs, ys)
    print(deg, "train RMSE", rmse(m, xs, ys),
          "test RMSE", rmse(m, big[["year"]], big["price_m2"]),
          "max|w|", f"{np.abs(m[-1].coef_).max():.3g}")

for a in [0.01, 1, 100]:
    m = poly(15, Ridge(alpha=a)).fit(xs, ys)
    print("ridge", a, "train", rmse(m, xs, ys),
          "test", rmse(m, big[["year"]], big["price_m2"]),
          "max|w|", f"{np.abs(m[-1].coef_).max():.3g}")

1 train RMSE 715 test RMSE 835 max|w| 1.16
4 train RMSE 685 test RMSE 880 max|w| 4.81e+06
15 train RMSE 684 test RMSE 885 max|w| 5.19e+06
ridge 0.01 train 711 test 842 max|w| 592
ridge 1 train 714 test 834 max|w| 59.2
ridge 100 train 715 test 834 max|w| 0.83


In [7]:
from sklearn.linear_model import RidgeCV, LassoCV

regs = {
    "ridge": RidgeCV(alphas=np.logspace(-2, 3, 30)),
    "lasso": LassoCV(alphas=np.logspace(0, 4, 30), cv=5,
                     random_state=42, max_iter=50000),
}
for name, reg in regs.items():
    m = make_pipeline(make_prep_bins(), reg)
    cv = cross_val_score(m, X_train, y_train, cv=5, scoring="r2").mean()
    m.fit(X_train, y_train)
    zeros = int((np.abs(m[-1].coef_) < 1e-6).sum())
    print(name, "alpha", round(m[-1].alpha_, 3), "cv", cv.round(3),
          "test", round(m.score(X_test, y_test), 3),
          "zeros", zeros, "of", len(m[-1].coef_))

lasso = m

ridge alpha 0.788 cv 0.914 test 0.912 zeros 0 of 19
lasso alpha 62.102 cv 0.914 test 0.912 zeros 2 of 19


In [8]:
base = cross_val_score(make_pipeline(make_prep(), Ridge(alpha=1)),
                       X_train, y_train, cv=5, scoring="r2").mean()
drop = {}
for col in num_cols + cat_cols:
    nn = [c for c in num_cols if c != col]
    cc = [c for c in cat_cols if c != col]
    prep = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), nn),
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore")), cc))
    s = cross_val_score(make_pipeline(prep, Ridge(alpha=1)),
                        X_train, y_train, cv=5, scoring="r2").mean()
    drop[col] = base - s

print(round(base, 3))
print(pd.Series(drop).sort_values(ascending=False).round(3))

0.916
area         0.150
district     0.140
year         0.027
furniture    0.004
floor       -0.000
rooms       -0.000
floors      -0.000
dtype: float64
